# BirdCLEF2026 — exp0000 推論Notebook

- **手法**: Perch v2 embedding (5秒チャンク) + `nn.Linear` head
- **OOF macro AUC (CV)**: 0.9901
- **使用Dataset**:
  - `birdclef2026-exp0000-weights` (学習済みヘッド `model.pt`)
  - `perch-onnx-for-birdclef-2026` (Perch v2 ONNX + onnxruntime wheel)
  - `birdclef-2026` (コンペデータ / test_soundscapes)

推論は **5秒チャンクごと** に行い、`row_id = <soundscape>_<終了秒>` で `submission.csv` を出力する。

In [ ]:
# 環境判定（Kaggle か ローカルか）してパスを切り替える
import os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None

if IS_KAGGLE:
    # kaggle CLI で push した kernel のマウント構成:
    #   コンペ          -> /kaggle/input/competitions/<slug>
    #   他者の Dataset  -> /kaggle/input/datasets/<owner>/<slug>
    #   自分の Dataset  -> /kaggle/input/<slug>
    COMP_DIR = Path('/kaggle/input/competitions/birdclef-2026')
    WEIGHTS_PATH = Path('/kaggle/input/birdclef2026-exp0000-weights/model.pt')
    PERCH_DIR = Path('/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026')
    PERCH_ONNX = PERCH_DIR / 'perch_v2.onnx'
    ONNX_WHEEL = PERCH_DIR / 'onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
else:
    # ローカル（リポジトリルートで実行する前提）
    COMP_DIR = Path('input')
    WEIGHTS_PATH = Path('experiments/exp0000/outputs/checkpoints/model.pt')
    PERCH_DIR = Path('input/perch_v2')
    PERCH_ONNX = PERCH_DIR / 'perch_v2_embedding_only.onnx'
    ONNX_WHEEL = PERCH_DIR / 'onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'

SAMPLE_SUB_PATH = COMP_DIR / 'sample_submission.csv'
TEST_AUDIO_DIR = COMP_DIR / 'test_soundscapes'
print('IS_KAGGLE =', IS_KAGGLE)
print('COMP_DIR  =', COMP_DIR)
print('PERCH_ONNX=', PERCH_ONNX)
print('WEIGHTS   =', WEIGHTS_PATH)

In [ ]:
# onnxruntime: 既存があれば利用、無ければ同梱wheelをオフラインインストール
import subprocess, sys
try:
    import onnxruntime  # noqa: F401
except ImportError:
    assert ONNX_WHEEL.exists(), f'onnxruntime wheel がありません: {ONNX_WHEEL}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', str(ONNX_WHEEL)], check=True)
    import onnxruntime  # noqa: F401

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import onnxruntime as ort

print('onnxruntime', ort.__version__, '| providers:', ort.get_available_providers())

SAMPLE_RATE = 32_000
CHUNK_SAMPLES = 160_000  # 5 s @ 32 kHz
EMBED_DIM = 1536
ONNX_BATCH = 64

# ラベル順は sample_submission の列順に厳密一致させる（学習時と同一の導出）
PRIMARY_LABELS = pd.read_csv(SAMPLE_SUB_PATH, nrows=1).columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
print('N_CLASSES =', N_CLASSES)

In [ ]:
# モデル定義（train.py と同一: nn.Linear(EMBED_DIM, N_CLASSES)）＋重みロード
model = nn.Linear(EMBED_DIM, N_CLASSES)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu'))
model.eval()

W = model.weight.detach().numpy().astype(np.float32)  # (N_CLASSES, EMBED_DIM)
B = model.bias.detach().numpy().astype(np.float32)     # (N_CLASSES,)

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def head_predict(emb):
    """emb: (n, EMBED_DIM) -> (n, N_CLASSES) softmax 確率"""
    return softmax(emb @ W.T + B)

In [ ]:
# 前処理 + Perch 埋め込み（cache builder と同一ロジック）
_providers = ort.get_available_providers()
_use = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in _providers else ['CPUExecutionProvider']
sess = ort.InferenceSession(str(PERCH_ONNX), providers=_use)
_in_name = sess.get_inputs()[0].name
_out_names = [o.name for o in sess.get_outputs()]
# フル perch_v2.onnx は複数出力。'embedding' 出力だけを要求する
_emb_name = 'embedding' if 'embedding' in _out_names else _out_names[0]
print('onnx in:', _in_name, '| using out:', _emb_name)

def load_waveform(path):
    wav, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return wav

def chunkify(wav):
    """1-D waveform -> (N, CHUNK_SAMPLES) 末尾ゼロパディング"""
    if wav.ndim != 1:
        wav = wav.reshape(-1)
    n = wav.shape[0]
    if n == 0:
        return np.zeros((1, CHUNK_SAMPLES), dtype=np.float32)
    n_chunks = (n + CHUNK_SAMPLES - 1) // CHUNK_SAMPLES
    total = n_chunks * CHUNK_SAMPLES
    if total > n:
        wav = np.pad(wav, (0, total - n))
    return wav.astype(np.float32, copy=False).reshape(n_chunks, CHUNK_SAMPLES)

def embed_chunks(chunks):
    """(N, CHUNK_SAMPLES) -> (N, EMBED_DIM)"""
    outs = []
    for i in range(0, len(chunks), ONNX_BATCH):
        emb = sess.run([_emb_name], {_in_name: chunks[i:i + ONNX_BATCH]})[0]
        outs.append(np.asarray(emb, dtype=np.float32))
    return np.concatenate(outs, axis=0).astype(np.float32)

In [ ]:
# 推論ループ: test_soundscapes の各音声を5秒チャンクごとに予測
AUDIO_EXTS = {'.ogg', '.wav', '.flac', '.mp3'}
test_files = sorted(p for p in TEST_AUDIO_DIR.rglob('*') if p.suffix.lower() in AUDIO_EXTS) if TEST_AUDIO_DIR.exists() else []
print('test files:', len(test_files))

rows = []
for path in test_files:
    stem = path.stem
    chunks = chunkify(load_waveform(path))
    probs = head_predict(embed_chunks(chunks))  # (n_chunks, N_CLASSES)
    for i in range(probs.shape[0]):
        end_sec = (i + 1) * 5
        row = {'row_id': f'{stem}_{end_sec}'}
        row.update({lbl: float(probs[i, j]) for j, lbl in enumerate(PRIMARY_LABELS)})
        rows.append(row)

print('predicted rows:', len(rows))

In [ ]:
# submission.csv 出力（列順は sample_submission に厳密一致）
cols = ['row_id'] + PRIMARY_LABELS
sub = pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols)
sub.to_csv('submission.csv', index=False)
print('saved submission.csv', sub.shape)
sub.head()